# 🔬  AI Data Concierge - Reproducible Analysis

<a href="https://colab.research.google.com/" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

---

## 📋 Query
> **Compare the predicited air quality inversions to the observed inversion data for allegheny county**

## 📊 Metadata
| Property | Value |
|----------|-------|
| **Generated** | 2026-05-12 20:02:32 |
| **Data Source** | WPRDC |
| **Sources Used** | Western PA Regional Data Center (WPRDC) |
| **Notebook Version** | 1.0 (Colab Compatible) |

---

## 📖 How to Use This Notebook

This notebook reproduces the exact analysis performed by the ** AI Data Concierge**.
Follow the steps below to verify, modify, or extend the analysis.

### ✅ Quick Start
1. **Run All Cells**: Click `Runtime` → `Run all` (or press `Ctrl+F9`)
2. **Wait for Setup**: The first cells install dependencies and configure the environment

### 🔧 What You Can Do
| Action | Description |
|--------|-------------|
| **Verify** | Run all cells to confirm the original results |
| **Modify** | Change parameters (dates, locations, filters) and re-run |
| **Extend** | Add your own analysis cells below the results |
| **Export** | Download results as CSV, or save notebook to Drive |

### 📚 Notebook Structure
1. **Setup** - Install dependencies (runs once in Colab)
2. **Configuration** - Import libraries and set up API connections
3. **Data Retrieval** - Fetch data from the data source
4. **Analysis** - Process and analyze the data
5. **Results** - View the final answer and confidence scores
6. **Citations** - Reference sources for your research

---


In [ ]:
# ============================================================
# STEP 1: Environment Setup
# ============================================================
# This cell installs all required packages for Google Colab.
# If running locally, you can skip this cell if packages are installed.

# Check if running in Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Running in Google Colab - Installing dependencies...")
    !pip install -q pandas numpy matplotlib seaborn requests
    print("✅ Dependencies installed successfully!")
else:
    print("💻 Running locally - assuming dependencies are installed")
    print("   If not, run: pip install pandas numpy matplotlib seaborn requests")


In [ ]:
# ============================================================
# STEP 2: Import Libraries
# ============================================================
# These are the core libraries used throughout this notebook.
# Each import serves a specific purpose in the data pipeline.

import requests      # For making HTTP requests to data APIs
import pandas as pd  # For data manipulation and analysis
import numpy as np   # For numerical computations
from datetime import datetime  # For timestamp handling
import json          # For JSON parsing

# Visualization libraries (with graceful fallback)
try:
    import matplotlib.pyplot as plt  # For creating plots
    import seaborn as sns            # For statistical visualizations
    VISUALIZATION_AVAILABLE = True
    plt.style.use('seaborn-v0_8-whitegrid')
    print("📊 Visualization libraries loaded successfully")
except ImportError:
    VISUALIZATION_AVAILABLE = False
    print("⚠️ Visualization libraries not available")
    print("   Install with: pip install matplotlib seaborn")

# ============================================================
# STEP 3: Configuration
# ============================================================
# Data source configuration - modify these if needed

CKAN_URL = "https://data.wprdc.org"

# Display configuration info
print(f"\n📅 Notebook generated: {datetime.now().isoformat()}")
print(f"🔗 CKAN URL: {CKAN_URL}")
print(f"📌 Timestamp: 2026-05-12T20:02:32.785822")


In [ ]:
# ============================================================
# STEP 4: Helper Functions
# ============================================================
# These utility functions handle data fetching from the CKAN API.
# You can reuse these functions for your own data exploration.

def fetch_ckan_data(resource_id: str, limit: int = 10000, filters: dict = None) -> pd.DataFrame:
    """
    Fetch data from CKAN DataStore API.

    This function handles pagination automatically and returns all records
    up to the specified limit.

    Parameters:
    -----------
    resource_id : str
        The unique identifier for the CKAN resource (dataset)
    limit : int, default=10000
        Maximum number of records to fetch
    filters : dict, optional
        Field filters to apply (e.g., {"state": "CA"})

    Returns:
    --------
    pd.DataFrame
        A DataFrame containing the fetched records

    Example:
    --------
    >>> df = fetch_ckan_data("abc123", limit=1000, filters={"year": 2023})
    >>> print(f"Loaded {len(df)} records")
    """
    print(f"📥 Fetching data from resource: {resource_id}")

    all_records = []
    offset = 0
    batch_size = min(32000, limit)

    while offset < limit:
        params = {
            "resource_id": resource_id,
            "limit": min(batch_size, limit - offset),
            "offset": offset,
        }

        if filters:
            params["filters"] = filters

        response = requests.post(
            f"{CKAN_URL}/api/3/action/datastore_search",
            json=params,
            headers={"Content-Type": "application/json"},
        )

        if response.status_code != 200:
            print(f"❌ Error: {response.status_code} - {response.text}")
            break

        result = response.json()
        if not result.get("success"):
            print(f"❌ CKAN error: {result.get('error')}")
            break

        records = result.get("result", {}).get("records", [])
        if not records:
            break

        all_records.extend(records)
        offset += len(records)

        total = result.get("result", {}).get("total", 0)
        print(f"   Progress: {len(all_records):,} / {total:,} records")

        if offset >= total:
            break

    df = pd.DataFrame(all_records)

    # Remove CKAN internal fields that aren't useful for analysis
    internal_cols = ["_id", "_full_text"]
    df = df.drop(columns=[c for c in internal_cols if c in df.columns], errors="ignore")

    print(f"✅ Loaded {len(df):,} records with {len(df.columns)} columns")
    return df


def search_ckan_packages(query: str, rows: int = 10) -> list:
    """
    Search CKAN for datasets (packages) matching a query.

    Parameters:
    -----------
    query : str
        The search query (e.g., "employment statistics")
    rows : int, default=10
        Maximum number of results to return

    Returns:
    --------
    list
        A list of matching package dictionaries

    Example:
    --------
    >>> packages = search_ckan_packages("housing prices", rows=5)
    >>> for pkg in packages:
    ...     print(pkg['title'])
    """
    print(f"🔍 Searching for: '{query}'")

    response = requests.post(
        f"{CKAN_URL}/api/3/action/package_search",
        json={"q": query, "rows": rows},
        headers={"Content-Type": "application/json"},
    )

    if response.status_code != 200:
        print(f"❌ Search failed: {response.status_code}")
        return []

    result = response.json()
    if not result.get("success"):
        print(f"❌ Search error: {result.get('error')}")
        return []

    packages = result.get("result", {}).get("results", [])
    print(f"✅ Found {len(packages)} matching datasets")
    return packages


print("✅ Helper functions loaded successfully!")
print("   - fetch_ckan_data(resource_id, limit, filters)")
print("   - search_ckan_packages(query, rows)")


# Data Analysis Pipeline

The following steps show how the data was searched, loaded, and analyzed. Each step includes executable code you can modify and re-run.

---

## Step 1: Search for Datasets

**Search query:** `air quality inversion predicted`
**Organization:** `city-of-pittsburgh`

**Result preview:**
```
Found 0 datasets matching 'air quality inversion predicted'

```


In [ ]:
# Step 1: Search for Datasets

# Search for datasets
import requests, json

params = {"q": 'air quality inversion predicted', "rows": 10, "fq": "organization:city-of-pittsburgh"}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 2: Search for Datasets

**Search query:** `air quality inversion allegheny county`

**Result preview:**
```
Found 6 datasets matching 'air quality inversion allegheny county'

1. **Allegheny County Air Quality**
   ID: `allegheny-county-air-quality`
   Air quality data is collected from the Allegheny County Health Department monitors throughout the county. This data must be verified by qualified individuals before it can be consi
   - Hourly Air Quality Data (CSV) [DataStore] ID: `36fb4629-8003-4acc-a1ca-3302778a530d`
   - Daily AQI Data (CSV) [DataStore] ID: `4ab1e23f-3262-4bd3-adbf-f72f0119108b`
   - Air Quality List of Acronyms and Flag Definitions (HTML) ID: `01297a0a-4160-41b3-8caf-90f7857ab7bd
```


In [ ]:
# Step 2: Search for Datasets

# Search for datasets
import requests, json

params = {"q": 'air quality inversion allegheny county', "rows": 10}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 3: Dataset Details

**Dataset:** `air-temperature-inversions`

**Result preview:**
```
# Air Temperature Inversions

This dataset contains the daily morning surface temperature inversion reports for the Pittsburgh, PA area. A temperature inversion occurs when the temperature increases with height as opposed to decreasing with height. Temperature inversions are common during the evening, nighttime, and morning hours. These inversions can sometimes result in poor air quality posing health risks especially for those who are immuno-compromised. To determine whether a surface temperature inversion is present, one must analyze a vertical profile of the atmosphere. The National Weather
```


In [ ]:
# Step 3: Dataset Details

# Get dataset details
resp = requests.get("https://data.wprdc.org/api/3/action/package_show", params={"id": 'air-temperature-inversions'})
ds = resp.json()["result"]

print(f"Title: {ds['title']}")
print(f"Description: {ds.get('notes', 'N/A')[:200]}")
print(f"\nResources:")
for r in ds.get("resources", []):
    act = "DataStore" if r.get("datastore_active") else "File"
    print(f"  - {r['name']} ({r['format']}, {act}) ID: {r['id']}")


## Step 4: Dataset Details

**Dataset:** `temperature-inversions`

**Result preview:**
```
# Temperature Inversions Forecasts

This dataset contains predictions of whether temperature inversions will occur at locations in Allegheny County. 

**This dataset is still under active development and should be considered to be in "beta".**

---
## Motivation ##
Temperature inversions occur when there is a warmer layer of air above the air at or near ground level. This represents a reversal of the normal flow of heat near the earth and results in the cooler air being trapped near the ground. Temperature inversions can lead to the formation of fog or dew. Pollution or smoke from fires,
```


In [ ]:
# Step 4: Dataset Details

# Get dataset details
resp = requests.get("https://data.wprdc.org/api/3/action/package_show", params={"id": 'temperature-inversions'})
ds = resp.json()["result"]

print(f"Title: {ds['title']}")
print(f"Description: {ds.get('notes', 'N/A')[:200]}")
print(f"\nResources:")
for r in ds.get("resources", []):
    act = "DataStore" if r.get("datastore_active") else "File"
    print(f"  - {r['name']} ({r['format']}, {act}) ID: {r['id']}")


## Step 5: Load Data from Resource

**Resource ID:** `34ec60b4-9ec7-40ad-9fa8-088cfcad8764`
**Limit:** 200

**Result preview:**
```
Resource: 34ec60b4-9ec7-40ad-9fa8-088cfcad8764
Total records: 1,947
Loaded: 200
Fields (6): date, inversion_y_n, temperature_c, depth_m, strength, notes

Sample (15 rows):

      date inversion_y_n temperature_c  depth_m strength notes
2026-05-08             Y           1.7    294.0     Weak  None
2026-05-07             Y           3.4     87.0 Moderate  None
2026-05-06             N           NaN      NaN      NaN  None
2026-05-05             N           NaN      NaN      NaN  None
2026-05-04             N           NaN      NaN      NaN  None
2026-05-03             Y           2.1    231.0  
```


In [ ]:
# Step 5: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '34ec60b4-9ec7-40ad-9fa8-088cfcad8764', "limit": 200, "sort": 'date desc'}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {\", \".join(df.columns.tolist())}")
df.head(10)


## Step 6: Load Data from Resource

**Resource ID:** `5ac4467e-a6de-4055-9111-9d83fb6584b4`
**Limit:** 200

**Result preview:**
```
Resource: 5ac4467e-a6de-4055-9111-9d83fb6584b4
Total records: 10
Loaded: 10
Fields (8): fp_forecast_day_s_, Predicted Strength, Total Days, Accuracy, True Positives, True Negatives, False Positives, False Negatives

Sample (10 rows):

fp_forecast_day_s_ Predicted Strength  Total Days  Accuracy  True Positives  True Negatives  False Positives  False Negatives
                5d                All         212    0.7170              68              84               31               29
                3d                All         212    0.7453              73              85               30     
```


In [ ]:
# Step 6: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5ac4467e-a6de-4055-9111-9d83fb6584b4', "limit": 200}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {\", \".join(df.columns.tolist())}")
df.head(10)


## Step 7: Load Data from Resource

**Resource ID:** `6a208a87-6ff1-4ee0-a7b4-6a201cdfaa69`
**Limit:** 100

**Result preview:**
```
Resource: 6a208a87-6ff1-4ee0-a7b4-6a201cdfaa69
Total records: 828
Loaded: 100
Fields (9): datetime, latitude_str, longitude_str, place, inversion, temp_diff, height, strength, forecast_version

Sample (15 rows):

           datetime latitude_str longitude_str place  inversion  temp_diff             height  strength forecast_version
2026-05-17T12:00:00         39.5        -81.25                1    1.19682 326.90369499999997         2    2026-05-12_00
2026-05-17T12:00:00         39.5      -80.9375                1    1.10307 326.90369499999997         2    2026-05-12_00
2026-05-17T12:00:00     
```


In [ ]:
# Step 7: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '6a208a87-6ff1-4ee0-a7b4-6a201cdfaa69', "limit": 100}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {\", \".join(df.columns.tolist())}")
df.head(10)


## Step 8: SQL Analysis Query

**SQL:**
```sql
SELECT "inversion_y_n", COUNT(*) as count, ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) as pct FROM "34ec60b4-9ec7-40ad-9fa8-088cfcad8764" WHERE "inversion_y_n" IS NOT NULL GROUP BY "inversion_y_n" ORDER BY count DESC
```

**Result preview:**
```
SQL: SELECT "inversion_y_n", COUNT(*) as count, ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) as pct FROM "34ec60b4-9ec7-40ad-9fa8-088cfcad8764" WHERE "inversion_y_n" IS NOT NULL GROUP BY "inversion_y_n" ORDER BY count DESC
Rows: 3
Columns: inversion_y_n, count, pct

inversion_y_n  count  pct
            N    980 51.4
            Y    927 48.6
            y      1  0.1
```


In [ ]:
# Step 8: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT "inversion_y_n", COUNT(*) as count, ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) as pct FROM "34ec60b4-9ec7-40ad-9fa8-088cfcad8764" WHERE "inversion_y_n" IS NOT NULL GROUP BY "inversion_y_n" ORDER BY count DESC'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 9: SQL Analysis Query

**SQL:**
```sql
SELECT "strength", COUNT(*) as count FROM "34ec60b4-9ec7-40ad-9fa8-088cfcad8764" WHERE "strength" IS NOT NULL AND "strength" != '' GROUP BY "strength" ORDER BY count DESC
```

**Result preview:**
```
SQL: SELECT "strength", COUNT(*) as count FROM "34ec60b4-9ec7-40ad-9fa8-088cfcad8764" WHERE "strength" IS NOT NULL AND "strength" != '' GROUP BY "strength" ORDER BY count DESC
Rows: 4
Columns: strength, count

strength  count
    Weak    319
Moderate    281
  Strong    243
  Slight     86
```


In [ ]:
# Step 9: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT "strength", COUNT(*) as count FROM "34ec60b4-9ec7-40ad-9fa8-088cfcad8764" WHERE "strength" IS NOT NULL AND "strength" != \'\' GROUP BY "strength" ORDER BY count DESC'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 10: SQL Analysis Query

**SQL:**
```sql
SELECT EXTRACT(MONTH FROM "date"::date) as month, COUNT(*) as total_days, SUM(CASE WHEN "inversion_y_n" = 'Y' THEN 1 ELSE 0 END) as inversion_days, ROUND(SUM(CASE WHEN "inversion_y_n" = 'Y' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as inversion_pct FROM "34ec60b4-9ec7-40ad-9fa8-088cfcad8764" WHERE "inversion_y_n" IS NOT NULL GROUP BY month ORDER BY month
```

**Result preview:**
```
SQL error (HTTP 403): {"help": "https://data.wprdc.org/api/3/action/help_show?name=datastore_search_sql", "error": {"__type": "Authorization Error", "message": "Access denied: Not authorized to call function EXTRACT"}, "success": false}
```


In [ ]:
# Step 10: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT EXTRACT(MONTH FROM "date"::date) as month, COUNT(*) as total_days, SUM(CASE WHEN "inversion_y_n" = \'Y\' THEN 1 ELSE 0 END) as inversion_days, ROUND(SUM(CASE WHEN "inversion_y_n" = \'Y\' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as inversion_pct FROM "34ec60b4-9ec7-40ad-9fa8-088cfcad8764" WHERE "inversion_y_n" IS NOT NULL GROUP BY month ORDER BY month'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 11: SQL Analysis Query

**SQL:**
```sql
SELECT DATE_PART('month', "date"::date) as month, COUNT(*) as total_days, SUM(CASE WHEN "inversion_y_n" = 'Y' THEN 1 ELSE 0 END) as inversion_days FROM "34ec60b4-9ec7-40ad-9fa8-088cfcad8764" WHERE "inversion_y_n" IS NOT NULL GROUP BY month ORDER BY month
```

**Result preview:**
```
SQL: SELECT DATE_PART('month', "date"::date) as month, COUNT(*) as total_days, SUM(CASE WHEN "inversion_y_n" = 'Y' THEN 1 ELSE 0 END) as inversion_days FROM "34ec60b4-9ec7-40ad-9fa8-088cfcad8764" WHERE "inversion_y_n" IS NOT NULL GROUP BY month ORDER BY month
Rows: 12
Columns: month, total_days, inversion_days

 month  total_days  inversion_days
   1.0         185              47
   2.0         151              55
   3.0         184              86
   4.0         175              86
   5.0         155              85
   6.0         149              82
   7.0         154              78
   8.0 
```


In [ ]:
# Step 11: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT DATE_PART(\'month\', "date"::date) as month, COUNT(*) as total_days, SUM(CASE WHEN "inversion_y_n" = \'Y\' THEN 1 ELSE 0 END) as inversion_days FROM "34ec60b4-9ec7-40ad-9fa8-088cfcad8764" WHERE "inversion_y_n" IS NOT NULL GROUP BY month ORDER BY month'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 12: SQL Analysis Query

**SQL:**
```sql
SELECT AVG("depth_m"::numeric) as avg_depth_m, AVG("temperature_c"::numeric) as avg_temp_diff_c, MIN("depth_m"::numeric) as min_depth, MAX("depth_m"::numeric) as max_depth FROM "34ec60b4-9ec7-40ad-9fa8-088cfcad8764" WHERE "inversion_y_n" = 'Y' AND "depth_m" IS NOT NULL
```

**Result preview:**
```
SQL: SELECT AVG("depth_m"::numeric) as avg_depth_m, AVG("temperature_c"::numeric) as avg_temp_diff_c, MIN("depth_m"::numeric) as min_depth, MAX("depth_m"::numeric) as max_depth FROM "34ec60b4-9ec7-40ad-9fa8-088cfcad8764" WHERE "inversion_y_n" = 'Y' AND "depth_m" IS NOT NULL
Rows: 1
Columns: avg_depth_m, avg_temp_diff_c, min_depth, max_depth

 avg_depth_m  avg_temp_diff_c  min_depth  max_depth
  231.928803          3.60109          8       1306
```


In [ ]:
# Step 12: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT AVG("depth_m"::numeric) as avg_depth_m, AVG("temperature_c"::numeric) as avg_temp_diff_c, MIN("depth_m"::numeric) as min_depth, MAX("depth_m"::numeric) as max_depth FROM "34ec60b4-9ec7-40ad-9fa8-088cfcad8764" WHERE "inversion_y_n" = \'Y\' AND "depth_m" IS NOT NULL'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


# ============================================================
# 📊 RESULTS
# ============================================================

## 💡 Answer

> **Now I have everything needed for a comprehensive comparison. Here is the full analysis:

---

# 🌡️ Predicted vs. Observed Air Quality Inversions — Allegheny County

**Sources:**
- **Observed:** *Air Temperature Inversions* dataset — daily morning balloon/sodar soundings by the Allegheny County Health Department (NWS Pittsburgh) — 1,947 records
- **Predicted:** *Temperature Inversions Forecasts* dataset (beta) — NASA GEOS-CF model — validated against 212 days of observations
- **Validation:** *Forecast Validation Statistics* resource (direct comparison table)

---

## 📊 1. Observed Inversion Summary (Historical Baseline)

| Metric | Value |
|---|---|
| Total observed days | 1,947 |
| Days **with** inversions (Y) | **927 (48.6%)** |
| Days **without** inversions (N) | 980 (51.4%) |
| Average inversion depth | **232 meters** |
| Average temperature difference | **3.6°C** |
| Deepest inversion recorded | 1,306 meters |
| Shallowest inversion recorded | 8 meters |

### Inversion Strength Breakdown (Observed)
| Strength | Days |
|---|---|
| 🟡 Weak | 319 |
| 🟠 Moderate | 281 |
| 🔴 Strong | 243 |
| 🔵 Slight | 86 |

---

## 📅 2. Seasonal Patterns (Observed — Monthly)

| Month | Total Days | Inversion Days | Rate |
|---|---|---|---|
| January | 185 | 47 | 25.4% |
| February | 151 | 55 | 36.4% |
| March | 184 | 86 | 46.7% |
| April | 175 | 86 | 49.1% |
| May | 155 | 85 | 54.8% |
| June | 149 | 82 | 55.0% |
| July | 154 | 78 | 50.6% |
| **August** | 147 | **90** | **61.2% ⬆ Peak** |
| **September** | 149 | **97** | **65.1% ⬆ Highest** |
| **October** | 155 | **93** | **60.0%** |
| November | 150 | 73 | 48.7% |
| December | 154 | 55 | 35.7% |

> **Key finding:** Inversions peak dramatically in **late summer and fall (Aug–Oct)**, with September being the worst month (65% of days have an inversion). Winter months (Dec–Jan) have the fewest — likely counterintuitive, but morning balloon soundings here reflect surface-layer dynamics specific to Pittsburgh's topography.

---

## 🤖 3. Forecast vs. Observed Accuracy (Validation Statistics)

This validation was performed over **212 days** of head-to-head comparison:

### All Inversion Types
| Forecast Lead Time | Accuracy | True Positives | True Negatives | False Positives | False Negatives |
|---|---|---|---|---|---|
| **1-day ahead** | **78.3%** | 73 | 93 | 22 | 24 |
| **3-day ahead** | **74.5%** | 73 | 85 | 30 | 24 |
| **5-day ahead** | **71.7%** | 68 | 84 | 31 | 29 |

### Strong/Moderate Inversions Only (most air quality-relevant)
| Forecast Lead Time | Accuracy | True Positives | False Positives |
|---|---|---|---|
| **1-day ahead** | **90.4%** | 47 | 5 |
| **3-day ahead** | **85.1%** | 40 | 7 |
| **5-day ahead** | **84.3%** | 43 | 8 |
| **5d + 3d + 1d combined** | **96.4%** | 27 | 1 |

---

## 🔍 4. Key Takeaways

| Finding | Detail |
|---|---|
| **Nearly half of all days have inversions** | 48.6% observed inversion rate across the full historical record |
| **Forecasts are quite accurate for the most dangerous events** | 90%+ accuracy for Strong/Moderate inversions at 1-day lead time |
| **False negatives are a concern for weak inversions** | 24 missed inversions at both 1-day and 3-day leads across 212 days |
| **Combined multi-day forecasts are most reliable** | Using 5d + 3d + 1d together hits 96.4% accuracy for Strong/Moderate events |
| **Fall is the highest-risk season** | September sees inversions on 65% of mornings |
| **Forecast skill degrades with lead time** | Accuracy drops ~6.6 percentage points from 1-day to 5-day forecast |

> ⚠️ **Caveat:** The *Temperature Inversions Forecasts* dataset is still labeled **"beta"** and under active development. The validation statistics are based on only 212 days of overlap. Observed data spans nearly 1,947 days of NWS balloon soundings, making it the more robust baseline.**

---

## 🎯 Confidence Assessment

The Data Concierge evaluates the reliability of its answer using multiple factors:


> ⚠️ Confidence score not available for this query.


# ============================================================
# 📚 CITATIONS & REFERENCES
# ============================================================

## 📖 Data Sources

## Data Sources and Citations

**[1]** Western PA Regional Data Center (WPRDC)
- Dataset: Open Data Portal
- URL: [https://data.wprdc.org](https://data.wprdc.org)
- Accessed: 2026-05-12

---

## 🔄 Reproducibility Guide

This notebook was automatically generated by the ** AI Data Concierge**.
Follow these steps to reproduce or extend the analysis:

### Prerequisites

```bash
pip install pandas numpy requests matplotlib seaborn
```

### Running the Notebook

| Step | Action | Notes |
|------|--------|-------|
| 1 | **Open in Colab** | Click the "Open in Colab" badge at the top |
| 2 | **Run All Cells** | `Runtime` → `Run all` or `Ctrl+F9` |
| 3 | **Wait for completion** | Dependencies install automatically in Colab |
| 4 | **Review results** | Scroll down to see the analysis results |

### ⚠️ Important Notes

- **Data freshness**: Results may differ if data sources have been updated since generation
- **API limits**: Some data sources have rate limits; wait if you encounter errors
- **Modifications**: Feel free to modify parameters and re-run cells to explore further

### 📅 Generation Info

- **Generated**: 2026-05-12 20:02:32
- **Query**: Compare the predicited air quality inversions to the observed inversion data for allegheny county
- **Data Source**: WPRDC

---

*Generated by  AI Data Concierge v0.1.0*
